In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/sample_submission.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/metadata.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv


# 3. FEATURE ENGINEERING

In [12]:
# ==============================================================================
# 3. FEATURE ENGINEERING
#    (a single function applied identically to train and test -> no leakage,
#     no duplicated logic)
# ==============================================================================
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # --- Date-based features -------------------------------------------------
    if "TransactionDate" in df.columns:
        df["TransactionDate"] = pd.to_datetime(df["TransactionDate"], errors="coerce")
        df["Txn_Year"] = df["TransactionDate"].dt.year
        df["Txn_Month"] = df["TransactionDate"].dt.month
        df["Txn_DayOfWeek"] = df["TransactionDate"].dt.dayofweek

        # Equipment age at time of transaction — a very strong price driver
        if "ManufactureYear" in df.columns:
            df["EquipmentAge"] = df["Txn_Year"] - df["ManufactureYear"]
            df["EquipmentAge"] = df["EquipmentAge"].clip(lower=0)

        df = df.drop(columns=["TransactionDate"])

    # --- Usage intensity: hours per year of age ------------------------------
    if {"OperationalHoursMeter", "EquipmentAge"}.issubset(df.columns):
        df["HoursPerYear"] = df["OperationalHoursMeter"] / df["EquipmentAge"].replace(0, 1)

    # --- Parse structured tokens out of the free-text spec descriptor --------
    if "Spec_FullDescriptor" in df.columns:
        df["Spec_FullDescriptor"] = df["Spec_FullDescriptor"].astype(str)
        df["Spec_TokenCount"] = df["Spec_FullDescriptor"].apply(lambda s: len(re.findall(r"[A-Za-z0-9]+", s)))
        # Pull out the first numeric token (often a horsepower / capacity figure)
        df["Spec_FirstNumber"] = df["Spec_FullDescriptor"].apply(
            lambda s: (lambda m: float(m.group()) if m else np.nan)(re.search(r"\d+(\.\d+)?", s))
        )

    # --- Generic anonymized spec columns (col1, col3, col4, ...) -------------
    generic_cols = [c for c in df.columns if re.fullmatch(r"col\d+", c)]
    if generic_cols:
        df["MissingSpecCount"] = df[generic_cols].isna().sum(axis=1)

    return df

train_fe = engineer_features(train)
test_fe = engineer_features(test)

print("After feature engineering:", train_fe.shape, test_fe.shape)


After feature engineering: (138701, 57) (15000, 56)


**Insight — feature engineering added 7 net columns and rebalanced numeric vs. categorical**

- Train grew from 50 → **57** columns, test from 49 → **56** — consistent (test lacks only `TargetValue`), confirming the same function ran cleanly on both without leaking train-only information.
- `MissingSpecCount` turns the missingness pattern flagged in Section 2.2 into a usable signal itself, rather than just something to impute around — if a machine is missing 8 of the anonymized spec columns because it's a different equipment type, that count is informative on its own.
- Net effect, confirmed in the next cell: numeric features roughly tripled (4 → 12) once `EquipmentAge`, `HoursPerYear`, `Spec_TokenCount`, `Spec_FirstNumber`, `MissingSpecCount` and the three date parts were added.


# 4. TRAIN / VALIDATION SPLIT

In [13]:
# ==============================================================================
# 4. TRAIN / VALIDATION SPLIT (log-transformed target, for RMSLE optimisation)
# ==============================================================================
feature_cols = [c for c in train_fe.columns if c not in [TARGET, ID_COL]]
# Keep only columns that also exist in test, to guarantee a valid final submission
feature_cols = [c for c in feature_cols if c in test_fe.columns]

X = train_fe[feature_cols]
y_log = np.log1p(train_fe[TARGET])

num_features = X.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X.select_dtypes(include=["object"]).columns.tolist()

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y_log, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Numeric features: {len(num_features)} | Categorical features: {len(cat_features)}")
print(f"Train: {X_train.shape} | Valid: {X_valid.shape}")


Numeric features: 12 | Categorical features: 43
Train: (110960, 55) | Valid: (27741, 55)


**Insight — 55 usable features, 80/20 split**

- **12 numeric + 43 categorical = 55** features feed the models (down from the raw 57 engineered columns, since `TargetValue` and `TransactionID` are excluded).
- `Train: (110,960, 55) | Valid: (27,741, 55)` — an 80/20 split on 138,701 rows. 27,741 validation rows is large enough that the RMSLE differences between models (e.g. 0.2137 vs 0.2169) are a meaningful comparison, not noise.
- Restricting `feature_cols` to columns present in *both* train and test (rather than trusting they'll always match) is what prevents a `KeyError` at prediction time in Section 15 — a small guard that avoids a late, hard-to-debug failure.


# 5. PREPROCESSING PIPELINE

In [14]:
# ==============================================================================
# 5. PREPROCESSING PIPELINE
#    (fit only on the training fold in every evaluation -> no leakage)
# ==============================================================================
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, num_features),
    ("cat", categorical_pipeline, cat_features),
])


**Insight — one `ColumnTransformer`, reused by every model**

- Median imputation for numerics is robust to the outliers visible in the Section 2.5 scatter plots; `"Missing"` as its own category for text columns keeps the informative-missingness pattern from Section 2.2 intact instead of erasing it.
- `OrdinalEncoder` (rather than one-hot) is the right choice given 43 categorical columns, several with 100+ unique values (`Spec_BaseClass` alone has 1,249) — one-hot encoding that would blow up to thousands of sparse columns. Tree-based models don't need one-hot's linear-separability assumption, so ordinal encoding loses nothing for XGBoost/LightGBM/CatBoost/Random Forest, and is only a minor handicap for Ridge.
- Because this whole block is wrapped in a `Pipeline`, it's refit fresh inside every `evaluate_model()` call on the training fold only — the validation and test sets never influence the imputer/encoder statistics.


# 6. EVALUATION HELPER

In [15]:
# ==============================================================================
# 6. EVALUATION HELPER (RMSLE, computed back on the original price scale)
# ==============================================================================
def rmsle_on_price_scale(y_true_log, y_pred_log):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log).clip(min=0)
    return np.sqrt(mean_squared_log_error(y_true, y_pred))

results = {}

def evaluate_model(name, pipeline):
    pipeline.fit(X_train, y_train)
    preds_valid = pipeline.predict(X_valid)
    rmsle = rmsle_on_price_scale(y_valid, preds_valid)
    mae = mean_absolute_error(np.expm1(y_valid), np.expm1(preds_valid).clip(min=0))
    r2 = r2_score(y_valid, preds_valid)
    results[name] = {"RMSLE": rmsle, "MAE": mae, "R2": r2}
    print(f"{name:22s} | RMSLE={rmsle:.4f} | MAE={mae:,.1f} | R2={r2:.4f}")
    return pipeline


**Insight — `.clip(min=0)` is a small line doing important work**

`mean_squared_log_error` is undefined for negative inputs, and a linear model like Ridge *can* predict a negative price for an extreme input combination. Clipping predictions to zero before the log is taken is what keeps this evaluation function from crashing on the weaker baseline model — without it, the very first `evaluate_model("Ridge (baseline)", ...)` call below could throw rather than report the 0.528 RMSLE it did.
